# script for parsing parliament speeches and selecting relevant ones

In [ ]:
#imports
import os
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import csv
from nltk.tokenize import RegexpTokenizer
import re
from IPython.display import display, HTML
import time   # for automatic pause
import spacy

tqdm.pandas()
import random
from IPython.core.display import display, HTML  # for Jupyter Notebook

#load the large Dutch model
nlp = spacy.load("nl_core_news_lg")

# classifier 
## code as used before

In [ ]:
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|LLaMA|openai|kunstmatige intelligentie systeem*|intelligente algoritme*|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drone|drones"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord"
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}

_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='text'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    # print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='text'):

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


# selecting relevant articles

In [ ]:
folder_path = r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Tweede Kamer\plenaire verslagen html"

rows = []
html_files = [f for f in os.listdir(folder_path) if f.endswith(".html")]

for idx, filename in enumerate(tqdm(html_files, desc="Parsing HTML files"), start=1):
    file_path = os.path.join(folder_path, filename)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    
    # Extract paragraphs
    paragraphs = [p.get_text() for p in soup.find_all('p')]
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    text_with_paragraphs = '\n'.join(paragraphs)
    
    # Create a temporary DataFrame with one row
    temp_df = pd.DataFrame([{'filename': filename, 'text': text_with_paragraphs}])
    
    # Apply classification
    classified_df = ai_classification(temp_df, title_col='filename', body_col='text')
    
    # Keep only if ai_related == 'yes'
    if classified_df['ai_related'].iloc[0] == 'yes':
        rows.append(classified_df.iloc[0])  # append the row
    
# Combine all rows into final DataFrame
df = pd.DataFrame(rows)

print("Done! DataFrame contains", len(df), "documents.")

#save to csv
df.to_csv'path', index=False)